<a href="https://colab.research.google.com/github/miriamamin1213-ux/HPV-classification/blob/main/Validation_on_Test_Reproducible.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Download the dataset from google drive

In [ ]:
#bash command/script in notebook
!gdown 1fPTHJb6LvvU3THZ2wAXPH3dUI-ziN0qh

#Train and test the model

In [ ]:
import os
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, balanced_accuracy_score,f1_score, roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score

from imblearn.over_sampling import SMOTE

def seed_everything(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8" # Required for newer PyTorch versions
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    np.random.seed(seed)
    random.seed(seed)


SEED = 42
seed_everything(seed = SEED)

df = pd.read_excel("HPVVAL25.xlsx")
df = df.dropna(subset=['HPV Status'])
tobacco_mode = df['Tobacco Consumption'].mode()[0]

df['Tobacco Consumption'] = df[
    'Tobacco Consumption'
].fillna(tobacco_mode)

alcohol_mode = df['Alcohol Consumption'].mode()[0]

df['Alcohol Consumption'] = df[
    'Alcohol Consumption'
].fillna(alcohol_mode)

df = df.drop(
    columns=[
        'PatientID',
        'CenterID',
        'Task 1',
        'Task 2',
        'Task 3'
    ]
)

df = df.dropna()
print(df.shape)

df['T-stage'] = df['T-stage'].replace({
    'T0':0,
    'T1':1,
    'T2':2,
    'T3':3,
    'T4':4
})

df['N-stage'] = df['N-stage'].replace({
    'N0':0,
    'N1':1,
    'N2':2,
    'N3':3
})

df['M-stage'] = df['M-stage'].replace({'M0':0,'M1':1})

X = df[
    [
        'Age',
        'Gender',
        'Tobacco Consumption',
        'Alcohol Consumption',
        'Performance Status',
        'Relapse',
        'RFS',
        'Treatment',
        'T-stage',
        'N-stage',
        'M-stage'
    ]
]

y = df['HPV Status']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED
)

print('train sample:',y_train.value_counts(),'\n')
print('test sample:', y_test.value_counts())

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
smote = SMOTE(random_state=SEED)
X_train_smote, y_train_smote = smote.fit_resample( X_train,y_train)
print('counting sample:', pd.Series(y_train_smote).value_counts())

#fixing data imbalance with smote
X_train_smote = torch.FloatTensor(X_train_smote)
y_train_smote = torch.LongTensor(y_train_smote.to_numpy())
X_test = torch.FloatTensor(X_test)
y_test = torch.LongTensor( y_test.to_numpy())

#Model Architecture
class HPVNet(nn.Module):

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(11,32)
        self.fc2 = nn.Linear(32,16)
        self.fc4 = nn.Linear(16,2)
        self.relu = nn.ReLU()

    def forward(self,x):
        # print('x:', x.shape)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc4(x)
        return x

#Defining model, and optimisation function/parameters
model = HPVNet()
weights = torch.tensor([3.0,1.0], dtype=torch.float32)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

#Training the model
epochs = 500
best_val_loss = float('inf')
best_epoch = 0

for epoch in range(epochs):
    model.train()
    outputs = model(X_train_smote)
    train_loss = criterion(outputs, y_train_smote)
    optimizer.zero_grad()
    train_loss.backward()
    optimizer.step()
    model.eval()

    with torch.no_grad():
        test_outputs = model(X_test)
        test_loss = criterion(test_outputs, y_test)

    if test_loss.item() < best_val_loss:

        best_val_loss = test_loss.item()
        best_epoch = epoch + 1
        torch.save(model.state_dict(), "best_model.pth")

    if (epoch+1) % 50 == 0:

        print(
            f"Epoch {epoch+1}, "
            f"Train={train_loss.item():.4f}, "
            f"Test={test_loss.item():.4f}"
        )

print("Best Test Loss =", best_val_loss)
print("Best Epoch =", best_epoch)


#Inferencing the mdoel
model.load_state_dict(torch.load("best_model.pth"))
model.eval()
with torch.no_grad():
    outputs = model(X_test)
    probabilities = torch.softmax(outputs, dim=1)  # Convert logits to probabilities
    predicted = torch.argmax(outputs,dim=1)

results = classification_report(y_test.numpy(),predicted.numpy(),digits=4)
print('results:',results)

# Balanced Accuracy
bal_acc = balanced_accuracy_score(y_test.numpy(),predicted.numpy())
f1 = f1_score(y_test.numpy(), predicted.numpy())

# auc = roc_auc_score(y_test.numpy(), probabilities.numpy())
y_prob = probabilities.cpu().numpy()
if y_prob.shape[1] == 2:   # Binary classification
    auc = roc_auc_score(y_test.numpy(), y_prob[:, 1])
else:                      # Multi-class classification
    auc = roc_auc_score(
        y_test.numpy(),
        y_prob,
        multi_class="ovr",
        average="weighted"
    )

print(f"Balanced Accuracy: {bal_acc:.4f}")
print(f"F1-score:          {f1:.4f}")
print(f"AUC:               {auc:.4f}")

(423, 12)
train sample: HPV Status
1.0    320
0.0     18
Name: count, dtype: int64 

test sample: HPV Status
1.0    80
0.0     5
Name: count, dtype: int64
counting sample: HPV Status
1.0    320
0.0    320
Name: count, dtype: int64
Epoch 50, Train=0.4135, Test=0.9871
Epoch 100, Train=0.2620, Test=0.7621


/tmp/ipykernel_5385/1727703751.py:60: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['T-stage'] = df['T-stage'].replace({
/tmp/ipykernel_5385/1727703751.py:68: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['N-stage'] = df['N-stage'].replace({
/tmp/ipykernel_5385/1727703751.py:75: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_si

Epoch 150, Train=0.1696, Test=0.6190
Epoch 200, Train=0.1287, Test=0.6203
Epoch 250, Train=0.1017, Test=0.6792
Epoch 300, Train=0.0806, Test=0.7357
Epoch 350, Train=0.0649, Test=0.7652
Epoch 400, Train=0.0521, Test=0.7959
Epoch 450, Train=0.0409, Test=0.8575
Epoch 500, Train=0.0316, Test=0.9293
Best Test Loss = 0.6111751198768616
Best Epoch = 172
results:               precision    recall  f1-score   support

           0     0.1429    0.6000    0.2308         5
           1     0.9688    0.7750    0.8611        80

    accuracy                         0.7647        85
   macro avg     0.5558    0.6875    0.5459        85
weighted avg     0.9202    0.7647    0.8240        85

Balanced Accuracy: 0.6875
F1-score:          0.8611
AUC:               0.8625


In [ ]:
#Archi: 11-32-16-2
Balanced Accuracy: 0.6875
F1-score:          0.8611
AUC:               0.8625